In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [2]:
data_dir = Path("data_mvg")
names = ['routes', 'stops', 'trips', 'stop_times', 'calendar']

tables = {name: pd.read_csv(data_dir / f'{name}.txt') for name in names}
routes, stops, trips, stop_times, calendar = tables.values()

In [3]:
for name, df in tables.items():
    print(f"{name}: {list(df.columns)}\n")
    print(f"shape: {df.shape}\n")

routes: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_color', 'route_text_color']

shape: (266, 8)

stops: ['stop_id', 'stop_name', 'stop_lat', 'stop_lon', 'location_type', 'parent_station']

shape: (4247, 6)

trips: ['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign', 'direction_id', 'block_id', 'route_direction']

shape: (53537, 8)

stop_times: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'pickup_type', 'drop_off_type', 'shape_dist_traveled']

shape: (1150460, 8)

calendar: ['service_id', 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday', 'start_date', 'end_date']

shape: (2, 10)



**Enforcing the rules of the meta.md file**

In [4]:
# Assign child stops to their parent stations

print(f"stops with a parent station: {stops['parent_station'].notnull().sum()}")
print(f"unique parent stations : {stops['parent_station'].nunique()}")

stops['station'] = stops['parent_station'].fillna(stops['stop_id'])

print(f"Unique effective stations: {stops['station'].nunique()} should equal parent stops (unique count)")

stops with a parent station: 3026
unique parent stations : 1221
Unique effective stations: 1221 should equal parent stops (unique count)


In [5]:
# Checks for clean data from the start so we won't have to deal with it later

# Times format
print("Sample times:")
print(stop_times[['arrival_time', 'departure_time']].head())

print(f"Max arrival_time: {stop_times['arrival_time'].max()}\n") # a number >24 means probably a time after midnight when the bus arrives at the stop while the bus starts its trip the day before
print(stop_times[['arrival_time', 'departure_time']].dtypes, "\n") # might need to convert to datetime later

# Distances
print(f"shape_dist_traveled nulls: {stop_times['shape_dist_traveled'].isna().sum()}\n")

# lat/lon completeness
print(f"stop_lat nulls: {stops['stop_lat'].isna().sum()}")
print(f"stop_lon nulls: {stops['stop_lon'].isna().sum()}\n")

# checking if stop times reference child stops
child_stops = stops[stops['parent_station'].notna()]['stop_id']
refs_to_children = stop_times['stop_id'].isin(child_stops).sum()
print(f"stop_times rows referencing child stops: {refs_to_children} / {len(stop_times)}")

Sample times:
  arrival_time departure_time
0     03:19:50       03:19:50
1     03:20:50       03:21:10
2     03:22:10       03:22:30
3     03:23:30       03:23:50
4     03:25:00       03:26:20
Max arrival_time: 27:43:40

arrival_time      str
departure_time    str
dtype: object 

shape_dist_traveled nulls: 0

stop_lat nulls: 0
stop_lon nulls: 0

stop_times rows referencing child stops: 1150460 / 1150460


In [6]:
# Map the child stops to their parent stations in the stop_times table
stop_id_to_station = stops.set_index('stop_id')['station']
stop_times['station'] = stop_times['stop_id'].map(stop_id_to_station)
print(f"Unmapped stations: {stop_times['station'].isna().sum()}")

Unmapped stations: 0


In [7]:
# Routes: variants vs. lines, and lines with more than one type
print(f"Unique route_id: {routes['route_id'].nunique()}")
print(f"Unique route_short_name: {routes['route_short_name'].nunique()}")

n_types = routes.groupby('route_short_name')['route_desc'].nunique()
conflicts = n_types[n_types > 1].index
print(f"Lines with more than one route_desc: {list(conflicts)}\n")

print(routes[routes['route_short_name'].isin(conflicts)]
      [['route_id', 'route_short_name', 'route_desc']]
      .sort_values('route_short_name'))

Unique route_id: 266
Unique route_short_name: 127
Lines with more than one route_desc: ['25', 'U6']

           route_id route_short_name             route_desc
68     2-25-G-016-1               25                   Tram
69     2-25-G-016-3               25                   Tram
70     2-25-G-016-4               25                   Tram
221  34-525-G-016-3               25  Schienenersatzverkehr
16     1-U6-G-016-1               U6                 U-Bahn
17    1-U6-G-016-12               U6                 U-Bahn
18    1-U6-G-016-13               U6                 U-Bahn
234  35-506-G-016-3               U6  Schienenersatzverkehr


**Lines with more than one route type**

Lines 25 and U6 appear both as their regular service and as a rail replacement bus
(*Schienenersatzverkehr*) under the same `route_short_name`. We do a check where these
variants actually call, to see how much this affects counting routes per station.

In [8]:
# route_ids of the conflicting lines, and how many trips each has
conflict_routes = routes.loc[routes['route_short_name'].isin(conflicts),
                             ['route_id', 'route_short_name', 'route_desc']]
conflict_trips = trips[['trip_id', 'route_id']].merge(conflict_routes, on='route_id')

print("Trips per conflicting route_id:")
print(conflict_trips['route_id'].value_counts()
      .reindex(conflict_routes['route_id'], fill_value=0), "\n")

# unique (station, line, type) combinations for these lines
conflict_pairs = (stop_times[['trip_id', 'station']]
                  .merge(conflict_trips[['trip_id', 'route_short_name', 'route_desc']], on='trip_id')
                  [['station', 'route_short_name', 'route_desc']]
                  .drop_duplicates())

# for each (line, station): which types go there?
types_at_station = conflict_pairs.groupby(['route_short_name', 'station'])['route_desc'].agg(set)

def assign(s):
    if len(s) == 2:
        return 'both'
    return 'SEV only' if s == {'Schienenersatzverkehr'} else 'regular only'

print(types_at_station.map(assign).groupby(level=0).value_counts().unstack(fill_value=0))

Trips per conflicting route_id:
route_id
1-U6-G-016-1      566
1-U6-G-016-12     841
1-U6-G-016-13       0
2-25-G-016-1      560
2-25-G-016-3      746
2-25-G-016-4        0
34-525-G-016-3    682
35-506-G-016-3      0
Name: count, dtype: int64 

route_desc        SEV only  both  regular only
route_short_name                              
25                       7     6            19
U6                       0     0            27


The U6 replacement route is listed in `routes.txt` but has no trips.
The line 25 replacement bus goes at 13 stations: 7 served only by the bus, 6 also served by the tram.

Following meta.md:
- **Route identifier:** `route_short_name`. Line 25 counts once per station, whether the tram, the bus, or both stop there.
- **Route type:** `route_desc` of the variant that actually goes at a station.

At the 6 shared stations, line 25 therefore contributes to both Tram and Schienenersatzverkehr,
so the per-type counts exceed the total by one. This affects 6 of 1221 stations and does not
change any distribution so we roll with this 